# Challenge 01 - PenDigits

Multiclass classification of handwritten digits (`0`-`9`) from 16 pen-trajectory features. Only `KNeighborsClassifier` or `SVC` are allowed, with scaling and PCA as the only preprocessing. The predictions are saved in `Es3Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Configuration
db_path = '../../DBs/PenDigits'
train_file = 'pendigits_tr.txt'
test_file = 'pendigits_te.txt'
result_path = '../Es3Predictions.txt'

feature_count = 16

## Helper Functions

Loaders for the PenDigits text files (16 features per row, followed by the label in the training file) and the export of the predictions.

In [2]:
def load_labeled_dataset(file_path, feature_count):
    data = np.loadtxt(file_path, usecols=range(0, feature_count))
    labels = np.loadtxt(file_path, usecols=[feature_count])
    return data, labels


def load_unlabeled_dataset(file_path, feature_count):
    return np.loadtxt(file_path, usecols=range(0, feature_count))


def save_predictions(predictions, path):
    """One integer label per line, no header."""
    with open(path, 'w') as f:
        for prediction in predictions:
            f.write(f'{int(prediction)}\n')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training file contains 16 input features and one digit label for each sample, while the test file contains the same 16 features without labels. No additional features are created: the original representation is passed to PCA and SVC.

In [3]:
train_x, train_y = load_labeled_dataset(os.path.join(db_path, train_file), feature_count)
test_x = load_unlabeled_dataset(os.path.join(db_path, test_file), feature_count)
print('Training samples: %d, test samples: %d, features: %d' % (len(train_x), len(test_x), train_x.shape[1]))

Training samples: 442, test samples: 5000, features: 16


## Model: PCA-SVC with Grid Search

An RBF-kernel SVC is used instead of KNN because it relies only on a subset of support vectors (faster inference, lower memory) and handles the non-linear decision boundaries of handwriting well. PCA, fitted on the training set only, reduces the dimensionality of the input. The number of PCA components, `C` and `gamma` are selected by a Grid Search; the `gamma` grid is a fine, geometrically spaced range around $10^{-4}$.

In [4]:
gammas = [float.fromhex(h) for h in (
    '0x1.0p-14', '0x1.4p-14', '0x1.8p-14', '0x1.cp-14',
    '0x1.0p-13', '0x1.4p-13', '0x1.8p-13', '0x1.cp-13',
    '0x1.0p-12'
)]

model = GridSearchCV(
    Pipeline([
        ('pca', PCA(random_state=42)),
        ('svc', SVC(kernel='rbf'))
    ]),
    param_grid={
        'pca__n_components': list(range(10, 17)),
        'svc__C': [1.5, 2.0, 2.8, 3.5, 5.0, 7.0],
        'svc__gamma': gammas
    },
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

## Model Evaluation

Each configuration is scored with stratified 5-fold cross-validation; the best one is then refitted on the whole training set.

In [5]:
model.fit(train_x, train_y)
print('Best parameters:', model.best_params_)
print('CV accuracy: %.4f' % model.best_score_)

Best parameters: {'pca__n_components': 16, 'svc__C': 1.5, 'svc__gamma': 0.000152587890625}
CV accuracy: 0.8439


## Prediction and Export

The selected pipeline predicts the digit of every test sample. The predictions are saved in `Es3Predictions.txt`, one integer label per line.

In [6]:
predictions = model.predict(test_x)
save_predictions(predictions, result_path)

Predictions successfully saved to ../Es3Predictions.txt
